# LayoutLMv3 with Auxiliary Boundary Loss (FUNSD & CORD — Base & Large)

This notebook sets up the exact baseline LayoutLMv3 environment on an **NVIDIA A100 GPU** in Google Colab, clones [`pn-minh/unilm`](https://github.com/pn-minh/unilm/tree/master/layoutlmv3), trains and evaluates **LayoutLMv3 (Base & Large)** with and without **Auxiliary Boundary Loss** on **FUNSD** and **CORD**, and syncs all metrics and logs directly to **Google Drive**.

## 1. Verify A100 GPU & Mount Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/thesis/layoutlmv3'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Google Drive output directory:', DRIVE_OUT)

## 2. Clone Fork & Bootstrap Exact Baseline Environment
Uses `scripts/setup_env.sh` to create the Python 3.7 conda environment with `torch==1.10.0+cu111`, `torchvision==0.11.1+cu111`, `transformers==4.12.5`, `datasets==1.15.1`, `timm==0.4.12`, `detectron2`, and installs `layoutlmft` in editable mode.

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /content/unilm/.git ]; then
  git clone --filter=blob:none --sparse https://github.com/pn-minh/unilm.git /content/unilm
  git -C /content/unilm sparse-checkout set layoutlmv3
else
  git -C /content/unilm pull --ff-only
fi
WORKDIR=/content REPO_URL=https://github.com/pn-minh/unilm.git bash /content/unilm/layoutlmv3/scripts/setup_env.sh

## 3. Smoke-Test Auxiliary Boundary Loss Implementation
Verifies that `LayoutLMv3ForTokenClassification` initializes `boundary_classifier`, extracts `B-*` entity-start targets (`1` for `B-*`, `0` for `I-*`/`O`, `-100` for special/visual/padding tokens), and computes the weighted auxiliary boundary loss.

In [ ]:
%%bash
source /root/miniconda3/etc/profile.d/conda.sh
conda activate layoutlmv3
cd /content/unilm/layoutlmv3

python - <<'EOF'
import torch
from layoutlmft.models.layoutlmv3 import LayoutLMv3Config, LayoutLMv3ForTokenClassification

# Test with FUNSD-like BIO scheme (7 labels, B-* at [1, 3, 5])
config = LayoutLMv3Config(
    vocab_size=100,
    hidden_size=64,
    num_hidden_layers=2,
    num_attention_heads=4,
    intermediate_size=128,
    max_position_embeddings=514,
    max_2d_position_embeddings=1024,
    coordinate_size=16,
    shape_size=16,
    visual_embed=False,
    num_labels=7,
    use_boundary_loss=True,
    boundary_loss_weight=0.2,
    boundary_pos_weight=5.0,
    b_label_ids=[1, 3, 5],
)
model = LayoutLMv3ForTokenClassification(config)
input_ids = torch.randint(0, 100, (2, 16))
bbox = torch.zeros((2, 16, 4), dtype=torch.long)
attention_mask = torch.ones((2, 16), dtype=torch.long)
labels = torch.tensor([
    [-100, 0, 1, 2, 3, 4, 0, 5, 6, 0, -100, -100, -100, -100, -100, -100],
    [-100, 1, 2, 0, 3, 0, 5, 6, 6, 0, -100, -100, -100, -100, -100, -100],
], dtype=torch.long)

out = model(input_ids=input_ids, bbox=bbox, attention_mask=attention_mask, labels=labels)
out.loss.backward()
print('Total loss:', out.loss.item())
print('Main NER loss:', model._last_ner_loss.item())
print('Boundary loss:', model._last_boundary_loss.item())
assert model.boundary_classifier.weight.grad is not None
print('Boundary Loss smoke test PASSED!')
EOF

## 4. Train & Evaluate on FUNSD (Base & Large)
Runs both baseline and Boundary Loss variants on **FUNSD** (`base` and `large`) and syncs each completed run directory to Google Drive.

In [ ]:
%%bash
set -euo pipefail
source /root/miniconda3/etc/profile.d/conda.sh
conda activate layoutlmv3
cd /content/unilm/layoutlmv3

export DRIVE_MOUNT=/content/drive/MyDrive
export DRIVE_DIR=thesis/layoutlmv3
export OUT=/content/runs

# FUNSD Base & Large (Baseline + Boundary Loss)
DATASETS="funsd" SIZES="base large" VARIANTS="baseline boundary" \
  BOUNDARY_LOSS_WEIGHT=0.2 BOUNDARY_POS_WEIGHT=5.0 \
  bash scripts/run_experiments.sh

## 5. Train & Evaluate on CORD (Base & Large)
Runs both baseline and Boundary Loss variants on **CORD** (`base` and `large`) and updates the aggregated summary on Google Drive.

In [ ]:
%%bash
set -euo pipefail
source /root/miniconda3/etc/profile.d/conda.sh
conda activate layoutlmv3
cd /content/unilm/layoutlmv3

export DRIVE_MOUNT=/content/drive/MyDrive
export DRIVE_DIR=thesis/layoutlmv3
export OUT=/content/runs
export PER_DEVICE=4

# CORD Base & Large (Baseline + Boundary Loss)
DATASETS="cord" SIZES="base large" VARIANTS="baseline boundary" \
  BOUNDARY_LOSS_WEIGHT=0.2 BOUNDARY_POS_WEIGHT=5.0 \
  bash scripts/run_experiments.sh

## 6. Inspect Aggregated Results Stored on Google Drive

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

summary_csv = '/content/drive/MyDrive/thesis/layoutlmv3/summary/summary_metrics.csv'
summary_md = '/content/drive/MyDrive/thesis/layoutlmv3/summary/summary_metrics.md'

if os.path.exists(summary_md):
    with open(summary_md, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
if os.path.exists(summary_csv):
    df = pd.read_csv(summary_csv)
    display(df)